# MERCon Dataset Audit Notebook

Run this notebook on Kaggle before running any evaluation. It does **not** rerun existing results. It only scans `/kaggle/input`, finds paired low/high image folders for the datasets mentioned by the reviewers, and prints the path variables to use later.

Reviewer-named datasets checked here: LOL-v2 Real, LOL-v2 Synthetic, LOL eval15, LOL-v1, SID, SMID, SDSD, VE-LOL, LSRW, ExDark.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
LOW_NAMES = {'low', 'input', 'inputs', 'lq', 'dark', 'darken', 'noisy', 'short'}
HIGH_NAMES = {'high', 'normal', 'gt', 'ground_truth', 'groundtruth', 'target', 'targets', 'long', 'hq'}

DATASET_KEYS = {
    'LOLV2_REAL': ['lol-v2', 'lol_v2', 'lolv2', 'real_captured', 'real'],
    'LOLV2_SYN': ['lol-v2', 'lol_v2', 'lolv2', 'synthetic', 'synth'],
    'EVAL15': ['eval15'],
    'LOLV1': ['lol-v1', 'lol_v1', 'lolv1', 'lol dataset', 'our485'],
    'SID': ['sid', 'see-in-the-dark', 'see_in_the_dark'],
    'SMID': ['smid'],
    'SDSD': ['sdsd'],
    'VELOL': ['ve-lol', 've_lol', 'velol'],
    'LSRW': ['lsrw'],
    'EXDARK': ['exdark', 'ex-dark'],
}

def count_images(path: Path) -> int:
    if not path.is_dir():
        return 0
    return sum(1 for p in path.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)

def direct_image_count(path: Path) -> int:
    try:
        return count_images(path)
    except Exception:
        return 0

def path_text(path: Path) -> str:
    return str(path).lower().replace('_', '-').replace(' ', '-')

def classify_dataset(path: Path) -> str:
    text = path_text(path)
    # Keep LOL-v2 Real/Synthetic disambiguation explicit.
    if any(k in text for k in ['lol-v2', 'lolv2', 'lol-v2']) and 'synthetic' in text:
        return 'LOLV2_SYN'
    if any(k in text for k in ['lol-v2', 'lolv2', 'lol-v2']) and ('real' in text or 'real-captured' in text):
        return 'LOLV2_REAL'
    for name, keys in DATASET_KEYS.items():
        if name in {'LOLV2_REAL', 'LOLV2_SYN'}:
            continue
        if any(k.replace('_', '-') in text for k in keys):
            return name
    return 'UNKNOWN'

def split_kind(path: Path) -> str:
    text = path_text(path)
    if 'train' in text or 'training' in text:
        return 'train'
    if 'test' in text or 'eval' in text or 'valid' in text or 'val' in text:
        return 'test'
    return 'unknown'

print('Kaggle input root exists:', INPUT_ROOT.exists())
print('Top-level Kaggle inputs:')
for p in sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
    print(' -', p)

In [ ]:
paired = []
image_only = []

for root, dirs, files in os.walk(INPUT_ROOT):
    root_path = Path(root)
    lower_to_dir = {d.lower(): d for d in dirs}

    low_child = next((lower_to_dir[n] for n in LOW_NAMES if n in lower_to_dir), None)
    high_child = next((lower_to_dir[n] for n in HIGH_NAMES if n in lower_to_dir), None)

    if low_child and high_child:
        low_dir = root_path / low_child
        high_dir = root_path / high_child
        n_low = count_images(low_dir)
        n_high = count_images(high_dir)
        if n_low > 0 and n_high > 0:
            paired.append({
                'dataset_guess': classify_dataset(root_path),
                'split_guess': split_kind(root_path),
                'parent': str(root_path),
                'low_dir': str(low_dir),
                'high_dir': str(high_dir),
                'n_low': n_low,
                'n_high': n_high,
                'paired_count_match': n_low == n_high,
            })

    n_here = direct_image_count(root_path)
    if n_here >= 10:
        image_only.append({
            'dataset_guess': classify_dataset(root_path),
            'split_guess': split_kind(root_path),
            'image_dir': str(root_path),
            'n_images': n_here,
        })

paired_df = pd.DataFrame(paired).sort_values(['dataset_guess', 'split_guess', 'n_low'], ascending=[True, True, False]) if paired else pd.DataFrame()
image_only_df = pd.DataFrame(image_only).sort_values(['dataset_guess', 'split_guess', 'n_images'], ascending=[True, True, False]) if image_only else pd.DataFrame()

print('Paired low/high candidates found:', len(paired_df))
display(paired_df)

print('Image-only folders found (possible no-reference datasets):', len(image_only_df))
display(image_only_df.head(80))

paired_df.to_csv(OUT_DIR / 'mercon_paired_dataset_candidates.csv', index=False)
image_only_df.to_csv(OUT_DIR / 'mercon_image_only_candidates.csv', index=False)
print('Wrote:', OUT_DIR / 'mercon_paired_dataset_candidates.csv')
print('Wrote:', OUT_DIR / 'mercon_image_only_candidates.csv')

In [ ]:
# Pick recommended test/eval paired folders for each reviewer-named dataset.
preferred = {}
if not paired_df.empty:
    for dataset in ['LOLV2_REAL', 'LOLV2_SYN', 'EVAL15', 'LOLV1', 'SID', 'SMID', 'SDSD', 'VELOL', 'LSRW', 'EXDARK']:
        sub = paired_df[paired_df['dataset_guess'] == dataset].copy()
        if sub.empty:
            continue
        # Prefer test/eval split, matching pair counts, and largest n.
        sub['split_rank'] = sub['split_guess'].map({'test': 0, 'unknown': 1, 'train': 2}).fillna(3)
        sub['match_rank'] = sub['paired_count_match'].map({True: 0, False: 1})
        sub = sub.sort_values(['split_rank', 'match_rank', 'n_low'], ascending=[True, True, False])
        row = sub.iloc[0].to_dict()
        preferred[dataset] = {
            'split_guess': row['split_guess'],
            'low_dir': row['low_dir'],
            'high_dir': row['high_dir'],
            'n_low': int(row['n_low']),
            'n_high': int(row['n_high']),
            'paired_count_match': bool(row['paired_count_match']),
        }

reviewer_datasets = ['LOLV2_REAL', 'LOLV2_SYN', 'EVAL15', 'LOLV1', 'SID', 'SMID', 'SDSD', 'VELOL', 'LSRW', 'EXDARK']
coverage_rows = []
for d in reviewer_datasets:
    info = preferred.get(d)
    coverage_rows.append({
        'dataset': d,
        'paired_candidate_found': info is not None,
        'split_guess': info.get('split_guess') if info else '',
        'n_low': info.get('n_low') if info else '',
        'n_high': info.get('n_high') if info else '',
        'low_dir': info.get('low_dir') if info else '',
        'high_dir': info.get('high_dir') if info else '',
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

coverage_df.to_csv(OUT_DIR / 'mercon_dataset_coverage_summary.csv', index=False)
with open(OUT_DIR / 'mercon_preferred_dataset_paths.json', 'w') as f:
    json.dump(preferred, f, indent=2)

print('Wrote:', OUT_DIR / 'mercon_dataset_coverage_summary.csv')
print('Wrote:', OUT_DIR / 'mercon_preferred_dataset_paths.json')

In [ ]:
# Print path assignments for kaggle_mercon_revision.sh.
var_prefix = {
    'LOLV2_REAL': 'LOLV2_REAL',
    'LOLV2_SYN': 'LOLV2_SYN',
    'EVAL15': 'EVAL15',
    'LOLV1': 'LOLV1',
    'SID': 'SID',
    'SMID': 'SMID',
    'SDSD': 'SDSD',
    'VELOL': 'VELOL',
    'LSRW': 'LSRW',
    'EXDARK': 'EXDARK',
}

lines = []
for dataset, prefix in var_prefix.items():
    info = preferred.get(dataset)
    if info:
        lines.append(f'{prefix}_LOW="{info["low_dir"]}"')
        lines.append(f'{prefix}_HIGH="{info["high_dir"]}"')
    else:
        lines.append(f'{prefix}_LOW=""')
        lines.append(f'{prefix}_HIGH=""')

snippet = '\n'.join(lines)
print(snippet)

with open(OUT_DIR / 'mercon_kaggle_path_assignments.sh', 'w') as f:
    f.write(snippet + '\n')

print('\nCopy these assignments into kaggle_mercon_revision.sh before running evaluations.')
print('Wrote:', OUT_DIR / 'mercon_kaggle_path_assignments.sh')

## Checkpoint detection

This cell finds `.pth`, `.pt`, and `.ckpt` files attached to Kaggle and recommends the checkpoint path to paste into `kaggle_mercon_revision.sh`.

In [ ]:
ckpt_exts = {'.pth', '.pt', '.ckpt'}
ckpt_rows = []
for p in INPUT_ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in ckpt_exts:
        name = p.name.lower()
        score = 0
        if 'distill' in name:
            score += 100
        if 'best' in name:
            score += 20
        if 'k5' in name or 'teacher-steps-5' in name:
            score += 10
        if name == 'final.pth':
            score += 5
        ckpt_rows.append({'path': str(p), 'filename': p.name, 'score': score})

ckpt_df = pd.DataFrame(ckpt_rows).sort_values(['score', 'filename'], ascending=[False, True]) if ckpt_rows else pd.DataFrame(columns=['path', 'filename', 'score'])
display(ckpt_df)
ckpt_df.to_csv(OUT_DIR / 'mercon_checkpoint_candidates.csv', index=False)

recommended_ckpt = ckpt_df.iloc[0]['path'] if len(ckpt_df) else ''
assignment = f'CHECKPOINT="{recommended_ckpt}"\n'
print('Recommended checkpoint assignment:')
print(assignment)
with open(OUT_DIR / 'mercon_checkpoint_assignment.sh', 'w') as f:
    f.write(assignment)

if not recommended_ckpt:
    print('No checkpoint found. Attach a Kaggle dataset containing final.pth or best_distill_K5.pth.')
elif 'distill' not in Path(recommended_ckpt).name.lower():
    print('Warning: recommended checkpoint name does not contain distill. Verify this is the FSD student checkpoint before using it for camera-ready results.')

print('Wrote:', OUT_DIR / 'mercon_checkpoint_candidates.csv')
print('Wrote:', OUT_DIR / 'mercon_checkpoint_assignment.sh')

## What to download

After this notebook finishes, download these files from `/kaggle/working`:

- `mercon_dataset_coverage_summary.csv`
- `mercon_paired_dataset_candidates.csv`
- `mercon_image_only_candidates.csv`
- `mercon_preferred_dataset_paths.json`
- `mercon_kaggle_path_assignments.sh`
- `mercon_checkpoint_candidates.csv`
- `mercon_checkpoint_assignment.sh`

Send those back or place them under `PAPER/revision_evidence/`. Then we will decide which missing datasets are worth evaluating. Do not rerun existing LOL-v2 Real results unless the path audit shows a mismatch.